# Practice 2 · Python environments
**Unit 1. Introduction to Machine Learning — 1.3. Methodologies and Work Practices**

### Objectives
By the end of this practice you will be able to:
1. Create an isolated Python environment with conda.
2. Install libraries in it.
3. Register the environment as a Jupyter kernel.
4. Check from a notebook which environment and libraries you are using.
5. Export the dependencies to `requirements.txt` so anyone can reproduce your work.

### Why environments?
Each project needs specific versions of its libraries. If every project shares the same Python installation, sooner or later one project's update breaks another. An **environment** is a separate space with its own Python and its own libraries:

| Project | Environment | pandas | scikit-learn |
|---|---|---|---|
| Project A | `env_a` | 2.1 | 1.3 |
| Project B | `env_b` | 2.2 | 1.5 |

### How to use this notebook
- Steps marked 🖥️ are done **in the terminal** (macOS/Linux) or the **Anaconda Prompt** (Windows).
- Cells marked ▶️ are run **here** to check that each step worked.
- Save this notebook in your repository, in the folder `practice_02/`.

> ⚠️ **macOS (zsh) users:** do not paste the `# comments` into the terminal. zsh does not treat `#` as a comment by default.

---
## 0. Before you start
You need **Anaconda** or **Miniconda** installed. Check it in the terminal 🖥️:
```bash
conda --version
```
- If you get `command not found: conda`, run `conda init zsh` (macOS) or open the **Anaconda Prompt** (Windows), then close and reopen the terminal.
- If conda is not installed: https://www.anaconda.com/download (or Miniconda for a lighter installation).

The prompt shows the active environment in brackets, for example `(base)`. **Do not install your project libraries in `base`.**

---
## 1. Create a Python environment 🖥️
```bash
conda create -n ml_env python=3.11
```
Answer `y` when asked to proceed. Then activate it:
```bash
conda activate ml_env
```
The prompt should change from `(base)` to `(ml_env)`.

Useful commands:
```bash
conda env list
conda deactivate
```
`conda env list` shows all your environments, with an asterisk on the active one. `conda deactivate` returns to `base`.

---
## 2. Install 5 libraries 🖥️
With `(ml_env)` active, install five core libraries for data science:

| Library | Use |
|---|---|
| `numpy` | Numerical computing with arrays |
| `pandas` | Tabular data (DataFrames) |
| `matplotlib` | Plots |
| `seaborn` | Statistical plots on top of matplotlib |
| `scikit-learn` | Machine learning models |

```bash
conda install numpy pandas matplotlib seaborn scikit-learn
```
You can also use `pip install numpy pandas matplotlib seaborn scikit-learn`.

> 💡 **conda or pip?** Use conda first. Use pip only for packages that conda does not have. Mixing both in the same environment works, but installing with conda *after* pip can break things.

You can choose other libraries; if you do, update the `LIBRARIES` list in step 4.

---
## 3. Install a kernel associated with the environment 🖥️
Jupyter runs code through a **kernel**. For a notebook to use `ml_env`, the environment must be registered as a kernel.

With `(ml_env)` active:
```bash
conda install ipykernel
python -m ipykernel install --user --name ml_env --display-name "Python (ml_env)"
```
- `--name` is the internal identifier.
- `--display-name` is the name you will see in Jupyter.

Check it:
```bash
jupyter kernelspec list
```

### Select the kernel in this notebook
- **Jupyter Notebook / JupyterLab:** *Kernel → Change kernel → Python (ml_env)*.
- **VS Code:** click the kernel selector (top right) → *Jupyter Kernel…* or *Python Environments…* → **Python (ml_env)**.

> ⚠️ The most common mistake is running `ipykernel install` **without activating the environment first**. The kernel is created, but it points to the wrong Python.

---
## 4. View the installed libraries in this notebook
### 4.1 Which Python is this notebook using?

In [ ]:
# ▶️ Python and environment used by this kernel
import os
import sys
from pathlib import Path

print("Python version :", sys.version.split()[0])
print("Executable     :", sys.executable)
print("Environment    :", os.environ.get("CONDA_DEFAULT_ENV") or Path(sys.prefix).name)

**Expected output:** the executable path contains `envs/ml_env`, for example `/Users/you/anaconda3/envs/ml_env/bin/python`.
If it shows `base` or another path, change the kernel (step 3) and run the cell again.

### 4.2 All installed libraries
`%pip` always uses the Python of the active kernel. `!pip` may use another one, so prefer `%pip`.

In [ ]:
# ▶️ Full list of packages in the environment
%pip list

### 4.3 Versions of your 5 libraries

In [ ]:
# ✍️ Your 5 libraries: package name -> import name
LIBRARIES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scikit-learn": "sklearn",
}

In [ ]:
# ▶️ Check that each library is installed and can be imported
import importlib
from importlib.metadata import version, PackageNotFoundError

rows = []
for package, module in LIBRARIES.items():
    try:
        v = version(package)
        importlib.import_module(module)
        status = "✅ OK"
    except PackageNotFoundError:
        v, status = "-", "❌ not installed"
    except ImportError as e:
        v, status = v, f"⚠️ import error: {e}"
    rows.append((package, v, status))

width = max(len(p) for p, _, _ in rows) + 2
print(f"{'Library':<{width}}{'Version':<12}Status")
print("-" * (width + 24))
for package, v, status in rows:
    print(f"{package:<{width}}{v:<12}{status}")

### 4.4 Quick test: use the libraries together
A short example that uses all five libraries: load a dataset, train a model and plot the result.

In [ ]:
# ▶️ Smoke test with the 5 libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

iris = load_iris(as_frame=True)
df = iris.frame.assign(species=iris.target_names[iris.target])

X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42, stratify=iris.target
)
model = LogisticRegression(max_iter=500).fit(X_train, y_train)
print(f"Test accuracy: {model.score(X_test, y_test):.2%}")
print(f"Mean petal length: {np.mean(df['petal length (cm)']):.2f} cm")

sns.scatterplot(data=df, x="petal length (cm)", y="petal width (cm)", hue="species")
plt.title("Iris dataset · environment working")
plt.show()

---
## 5. Run `pip freeze > requirements.txt`
`pip freeze` lists every installed package with its exact version (`package==version`). The `>` symbol writes that list to a file instead of the screen.

**Option A. Terminal 🖥️** (with `(ml_env)` active, in the root folder of your repository):
```bash
pip freeze > requirements.txt
```

**Option B. From this notebook ▶️** (next cell). It uses the kernel's Python and saves the file in the root of your repository.

In [ ]:
# ▶️ Generate requirements.txt in the root of the repository
import subprocess

top = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
repo_root = Path(top.stdout.strip()) if top.returncode == 0 else Path.cwd()
req_file = repo_root / "requirements.txt"

frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
req_file.write_text(frozen, encoding="utf-8")

lines = [l for l in frozen.splitlines() if l and not l.startswith("#")]
print(f"Saved {len(lines)} packages to {req_file}\n")
print("\n".join(lines[:15]), "\n..." if len(lines) > 15 else "")

### ⚠️ Lines with `@ file:///` (conda environments)
In conda environments, `pip freeze` sometimes writes local paths instead of versions, for example:
```
numpy @ file:///Users/builder/.../numpy-1.26.4.whl
```
These lines **only work on your computer**. The next cell checks for them and, if needed, regenerates the file with `pip list --format=freeze`, which always writes `package==version`.

In [ ]:
# ▶️ Check for local paths and fix them if needed
content = req_file.read_text(encoding="utf-8")
if " @ file://" in content:
    fixed = subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"],
                           capture_output=True, text=True).stdout
    req_file.write_text(fixed, encoding="utf-8")
    print("Local paths found → regenerated with 'pip list --format=freeze' ✅")
else:
    print("No local paths found ✅")

### Reproduce the environment
Anyone (or you, on another computer) can recreate the environment with:
```bash
conda create -n ml_env python=3.11
conda activate ml_env
pip install -r requirements.txt
```
For conda users there is also an equivalent export that lists only the packages you explicitly installed:
```bash
conda env export --from-history > environment.yml
```

---
## 6. Upload the practice to GitHub 🖥️
As in Practice 1, from the root of your repository:
```bash
git pull
git add practice_02/ requirements.txt
git commit -m "Add practice 2: Python environments"
git push
```
Or use **Source Control** in VS Code: Commit → Push (↑).

---
## Troubleshooting
| Problem | Cause | Solution |
|---|---|---|
| `command not found: conda` | conda is not initialised in your shell | `conda init zsh` (macOS) and reopen the terminal; on Windows use the Anaconda Prompt |
| `ModuleNotFoundError` in the notebook | The notebook uses another kernel | Select **Python (ml_env)** and check `sys.executable` (step 4.1) |
| The kernel does not appear in Jupyter / VS Code | Not registered, or the editor did not refresh | `jupyter kernelspec list`; restart Jupyter or run *Developer: Reload Window* in VS Code |
| The kernel exists but uses the wrong Python | `ipykernel install` was run outside the environment | `conda activate ml_env`, then repeat step 3 |
| `@ file:///` in `requirements.txt` | Conda-built packages | Use `pip list --format=freeze > requirements.txt` |
| Libraries installed in `base` | The environment was not active | `conda activate ml_env` and install again |
| Remove a kernel or environment | — | `jupyter kernelspec uninstall ml_env` · `conda env remove -n ml_env` |

---
## ✅ Deliverable
Run the final check, save the notebook and push it to GitHub.

In [ ]:
# ▶️ Final check
import json

env_name = os.environ.get("CONDA_DEFAULT_ENV") or Path(sys.prefix).name
in_env = env_name != "base" and ("envs" in Path(sys.executable).parts or sys.prefix != sys.base_prefix)

ks = subprocess.run([sys.executable, "-m", "jupyter", "kernelspec", "list", "--json"],
                    capture_output=True, text=True)
kernels = json.loads(ks.stdout).get("kernelspecs", {}) if ks.returncode == 0 else {}
kernel_ok = any(env_name in name or env_name in json.dumps(spec) for name, spec in kernels.items())

installed = [p for p in LIBRARIES if importlib.util.find_spec(LIBRARIES[p]) is not None]

req_text = req_file.read_text(encoding="utf-8").lower() if req_file.exists() else ""
in_req = [p for p in LIBRARIES if f"{p.lower()}==" in req_text]

checks = {
    f"1. Notebook running in an environment other than base ({env_name})": in_env,
    "2. 5 libraries installed and importable": len(installed) >= 5,
    "3. Kernel registered in Jupyter for this environment": kernel_ok,
    "4. Libraries listed in this notebook (step 4 executed)": "rows" in globals(),
    "5. requirements.txt exists with the 5 libraries": len(in_req) >= 5,
    "5. requirements.txt without local paths": req_file.exists() and " @ file://" not in req_text,
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)

**Submit:** push this notebook (`assignment_02/`) and `requirements.txt` to your GitHub repository.